# AI Laboratory: Logical Planning
**Course:** CS F407 - Artificial Intelligence  
**Notebook:** `logical_planning_notebook.ipynb`  
**Reference Document:** `logic_lab_ex.pdf`

This notebook implements a STRIPS-style planning agent combining Propositional Logic with Breadth-First Search (BFS), accompanied by an embedded Prolog plan verifier.

## 1. Planning Problem Definition (Warehouse Scenario)
- Locations: $A, B, C$
- Robot and Package start at $A$
- Goal: Package at $C$
- Formula: $\text{Logic} + \text{Search} = \text{Planning}$

In [1]:
from logical_planner import create_warehouse_actions, PlanningProblem

initial_state = {'At(Robot, A)', 'At(Package, A)'}
goal_state = {'At(Package, C)'}
actions = create_warehouse_actions(allow_pickup=True)

problem = PlanningProblem(initial_state, actions, goal_state)
print(f'Initial State: {initial_state}')
print(f'Goal State: {goal_state}')
print(f'Available Actions: {len(actions)}')


## 2. Test A: Solvable Problem Execution (BFS Plan Search)
Finding the optimal plan and tracing step-by-step state progression.

In [2]:
result = problem.solve_bfs()
print(f'Plan Found: {result["success"]}')
print(f'Plan Length: {len(result["plan"])} steps\n')
for i, act in enumerate(result['plan'], 1):
    print(f'Step {i}: {act}')

print('\nState Transitions:')
for i, st in enumerate(result['states']):
    print(f'S_{i}: {sorted(list(st))}')


## 3. Test B & C: Negative and Irrelevant Action Oracles
Validating boundary conditions: impossible problems and spurious action resistance.

In [3]:
# Test B: Impossible Problem
actions_no_pickup = create_warehouse_actions(allow_pickup=False)
prob_b = PlanningProblem(initial_state, actions_no_pickup, goal_state)
res_b = prob_b.solve_bfs()
print(f'Test B (No PickUp): Plan Found? {res_b["success"]} (Expected: False)')

# Test C: Irrelevant Actions
actions_c = create_warehouse_actions(allow_pickup=True, include_irrelevant=True)
prob_c = PlanningProblem(initial_state, actions_c, goal_state)
res_c = prob_c.solve_bfs()
print(f'Test C (With Irrelevant Actions): Plan Found? {res_c["success"]} (Length: {len(res_c["plan"])})')


## 4. Prolog as an Independent Plan Verifier
Using declarative rules and Horn clause resolution to independently check movement validity.

In [4]:
from logical_planner import PrologVerifier

prolog = PrologVerifier()
for u, v in [('a', 'b'), ('b', 'a'), ('b', 'c'), ('c', 'b')]:
    prolog.add_fact('connected', u, v)
prolog.add_rule(('valid_move', 'X', 'Y'), [('connected', 'X', 'Y')])

print('Verifying Plan Steps:')
print('Step 1 (a -> b):', prolog.query('valid_move', 'a', 'b'))
print('Step 2 (b -> c):', prolog.query('valid_move', 'b', 'c'))
print('Invalid Step (a -> c):', prolog.query('valid_move', 'a', 'c'))
